# Práctica 3: Ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [NLTK](https://www.nltk.org) de Python.

### Ejercicio 1

El objetivo de este ejercicio es entrenar y evaluar el rendimiento de un filtro de correo electrónico no deseado. Para ello se usará el corpus Enron-Spam, pero no se proporcionará un vocabulario fijo, sino que este deberá aprenderse a partir de los mensajes de entrenamiento. Con el objetivo de homogeneizar el vocabulario aprendido y de mejorar el rendimiento del filtro construido, se pedirá que se apliquen distintas técnicas de preprocesado.

En todos los apartados de este ejercicio se deberá realizar lo siguiente:

* Construir el filtro como una tubería de scikit-learn que concatene un vectorizador tf-idf y un modelo $k$NN clasificador con 5 vecinos y que use la métrica del coseno.
* Definir una función `procesa_mensaje` que, dado el contenido en bruto de un mensaje, aplique todos los pasos de procesamiento pedidos hasta obtener la lista de tókenes correspondiente. Esta función se deberá proporcionar como argumento `analyzer` del vectorizador tf-idf.
* Entrenar el filtro con el corpus de entrenamiento.
* Calcular la sensibilidad del filtro sobre el corpus de prueba.

In [ ]:
from email import parser
from email import policy

In [ ]:
analizador_mensaje = parser.Parser(policy=policy.default)

In [ ]:
from pathlib import Path

In [ ]:
carpeta_Enron_Spam = Path('Filtro antispam/Enron-Spam/')
carpeta_entrenamiento = carpeta_Enron_Spam / 'train'
carpeta_prueba = carpeta_Enron_Spam / 'test'

contenidos_mensajes_entrenamiento = []
clases_mensajes_entrenamiento = []
for ruta_mensaje in (carpeta_entrenamiento / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_entrenamiento / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_entrenamiento.append(mensaje.get_content())
            clases_mensajes_entrenamiento.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

contenidos_mensajes_prueba = []
clases_mensajes_prueba = []
for ruta_mensaje in (carpeta_prueba / 'legítimo').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(0)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass
for ruta_mensaje in (carpeta_prueba / 'no_deseado').iterdir():
    with open(ruta_mensaje, 'r') as fichero_mensaje:
        try:
            mensaje = analizador_mensaje.parse(fichero_mensaje)
            contenidos_mensajes_prueba.append(mensaje.get_content())
            clases_mensajes_prueba.append(1)
        except (KeyError, UnicodeDecodeError, LookupError):
            pass

#### Apartado 0

En este apartado se pide procesar los mensajes realizando los siguientes 3 pasos:

* Extraer el contenido de texto de los mensajes en formato HTML. Para ello hacer uso de la biblioteca [Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/).
* Dividir el contenido de los mensajes en secuencias de tókenes mediante el tokenizador de NLTK.
* Eliminar de los tókenes los caracteres no alfanuméricos (y eliminar por completo aquellos tókenes que no contengan caracteres alfanuméricos).

In [ ]:
contenidos_mensajes_entrenamiento[-22]

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
def elimina_html(contenido):
    return BeautifulSoup(contenido).get_text()

In [ ]:
elimina_html(contenidos_mensajes_entrenamiento[-22])

In [ ]:
import os

os.environ['NLTK_DATA'] = '.'

In [ ]:
from nltk import download

download('punkt_tab')

In [ ]:
from nltk.tokenize import word_tokenize

In [ ]:
from pprint import pprint

In [ ]:
pprint(word_tokenize(elimina_html(contenidos_mensajes_entrenamiento[-22])),
       compact=True)

La eliminación de los caracteres no alfanuméricos se puede realizar mediante expresiones regulares, usando para ello el paquete [re](https://docs.python.org/es/3/library/re.html) de la biblioteca estándar de Python.

In [ ]:
import re

In [ ]:
def elimina_no_alfanumerico(contenido):
    return [re.sub(r'[^\w]', '', palabra)
            for palabra in contenido
            if re.search(r'\w', palabra)]

In [ ]:
def procesa_mensaje(contenido):
    contenido = elimina_html(contenido)
    contenido = word_tokenize(contenido)
    contenido = elimina_no_alfanumerico(contenido)
    return contenido

In [ ]:
pprint(procesa_mensaje(contenidos_mensajes_entrenamiento[-22]),
       compact=True)

Debido a la naturaleza de los mensajes no deseados, algunos de ellos pueden confundir a la biblioteca Beautiful Soup, avisando esta de que el mensaje puede tratarse de una URL o de una ruta a un fichero, en lugar de un mensaje de correo electrónico. El código de la siguiente celda filtra ese tipo de avisos.

In [ ]:
from bs4 import MarkupResemblesLocatorWarning
import warnings

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

Estamos ya en condiciones de poder construir el filtro de correo electrónico no deseado.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
filtro_antispam = Pipeline([
    ('vectorizador', TfidfVectorizer(analyzer=procesa_mensaje)),
    ('modelo', KNeighborsClassifier(n_neighbors=5, metric='cosine'))
])

In [ ]:
filtro_antispam.fit(contenidos_mensajes_entrenamiento,
                    clases_mensajes_entrenamiento)

In [ ]:
from sklearn.metrics import recall_score

In [ ]:
predicciones_mensajes_prueba = filtro_antispam.predict(
    contenidos_mensajes_prueba
)
recall_score(clases_mensajes_prueba, predicciones_mensajes_prueba)

#### Apartado 1

En este apartado se pide incorporar al procesado de mensajes los siguientes 2 pasos:

* Expandir las contracciones típicas del idioma inglés. Usar para ello el paquete [contractions](https://github.com/kootenpv/contractions).
* Convertir todos los caracteres a minúsculas.

#### Apartado 2

Palabras vacías (_stop words_, en inglés) es el nombre que reciben las palabras tales como artículos, pronombres y preposiciones que se considera que no aportan significado para un sistema de procesamiento del lenguaje natural y que, por tanto, deben eliminarse durante las operaciones de preprocesado de texto. El conjunto adecuado de palabras vacías a usar depende del sistema concreto que se esté construyendo, e incluso puede resultar conveniente no hacer uso de esta técnica.

NLTK provee de conjuntos genéricos de palabras vacías para distintos idiomas.

In [ ]:
download('stopwords')

In [ ]:
from nltk.corpus import stopwords

In [ ]:
palabras_vacias_ingles = stopwords.words('english')
pprint(palabras_vacias_ingles, compact=True)

En este apartado se pide incorporar al procesado de mensajes la eliminación de palabras vacías.

#### Apartado 3

Por razones gramaticales, en un documento de texto van a aparecer con seguridad diferentes formas de una palabra, como organizar, organiza y organizando. Además, existen familias de palabras relacionadas derivativamente con significados similares, como democracia, democrático y democratización. En muchas situaciones, parece que sería útil reducir esos conjuntos de palabras a una raíz común. Para ello se suelen usar los procedimientos de _stemming_ y lematización.

_Stemming_ generalmente se refiere a un proceso heurístico rudimentario que corta los extremos de las palabras con la esperanza de lograr el objetivo correctamente la mayor parte del tiempo y, a menudo, incluye la eliminación de afijos derivativos. La lematización generalmente se refiere a hacer las cosas correctamente con el uso de un vocabulario y análisis morfológico de las palabras, normalmente con el objetivo de eliminar únicamente las terminaciones flexivas y devolver la forma base o de diccionario de una palabra, lo que se conoce como lema.

NLTK provee de varios algoritmos de _stemming_ y lematización. En este apartado se pide incorporar al procesado de mensajes el procedimiento de _stemming_ mediante el [algoritmo de Lancaster](https://www.nltk.org/api/nltk.stem.lancaster.html).

In [ ]:
from nltk.stem.lancaster import LancasterStemmer

### Ejercicio 2

En el cuaderno NLTK.ipynb se ha construido un sistema de predicción de texto en español basado en modelos de $n$-gramas. Estos modelos se han entrenado a partir de un corpus de textos en español que se ha usado en bruto. El objetivo de este ejercicio es recrear la construcción del sistema de predicción de texto, pero usando una versión normalizada del corpus.

#### Apartado 1

En este apartado se pide:

1. Leer el corpus guardado en el fichero `Texto predictivo/corpus_InfoLibros_parcial.txt` y dividirlo en un corpus de entrenamiento y un corpus de prueba.
2. Construir modelos unigramas, bigramas y trigramas, con y sin suavizado, a partir del corpus de entrenamiento normalizado convirtiendo todas las palabras a minúsculas.
3. Seleccionar el modelo con menor perplejidad sobre el corpus de prueba normalizado convirtiendo todas las palabras a minúsculas.

#### Apartado 2

Redefinir la función `predice_palabras` de tal forma que prediga, a partir de las palabras anteriores y de las letras de la palabra ya escritas, qué palabra se pretende escribir, actuando como sigue:

* Si todas las letras del prefijo escrito están en minúsculas, entonces debe predecir palabras en minúsculas.
* Si todas las letras del prefijo escrito están en mayúsculas, entonces debe predecir palabras en mayúsculas.
* Si el prefijo escrito mezcla letras en minúsculas y en mayúsculas, entonces:
  * Si la primera letra del prefijo está en minúsculas, entonces debe predecir palabras en minúsculas.
  * Si la primera letra del prefijo está en mayúsculas, entonces debe predecir palabras con la primera letra en mayúsculas y el resto en minúsculas.

In [ ]:
predice_palabras('nat', ('Lenguaje',), 5, modelo_bigrama_Laplace)
# ['natural', 'naturalmente', 'nata', 'naturales', 'natacha']

In [ ]:
predice_palabras('NAT', ('Lenguaje',), 5, modelo_bigrama_Laplace)
# ['NATURAL', 'NATURALMENTE', 'NATA', 'NATURALES', 'NATACHA']

In [ ]:
predice_palabras('nAt', ('Lenguaje',), 5, modelo_bigrama_Laplace)
# ['natural', 'naturalmente', 'nata', 'naturales', 'natacha']

In [ ]:
predice_palabras('NaT', ('Lenguaje',), 5, modelo_bigrama_Laplace)
# ['Natural', 'Naturalmente', 'Nata', 'Naturales', 'Natacha']